In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm opencv-python-headless

import os, json, random, copy, time, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from concurrent.futures import ThreadPoolExecutor
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

TEA_LEAF_ROOT = "/content/drive/MyDrive/Tea Leaf Dataset"
OUT           = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
RESULTS_DIR   = f"{OUT}/results"
FIGURES_DIR   = f"{OUT}/figures"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# ---- dataset -> local disk (parallel, resumable) ----
LOCAL_DATA = "/tmp/tea_leaf_local"
N_WORKERS  = 32

def _count_files(p):
    return sum(len(f) for _, _, f in os.walk(p)) if os.path.isdir(p) else 0

n_src = _count_files(TEA_LEAF_ROOT)
n_dst = _count_files(LOCAL_DATA)

if n_dst < n_src:
    t0 = time.time()
    tasks = []
    for root, _, files_ in os.walk(TEA_LEAF_ROOT):
        rel = os.path.relpath(root, TEA_LEAF_ROOT)
        dst_dir = LOCAL_DATA if rel == '.' else os.path.join(LOCAL_DATA, rel)
        os.makedirs(dst_dir, exist_ok=True)
        for f in files_:
            src, dst = os.path.join(root, f), os.path.join(dst_dir, f)
            if not os.path.exists(dst):
                tasks.append((src, dst))
    print(f"Copying {len(tasks)} files with {N_WORKERS} workers...")
    def _cp(t):
        try:
            shutil.copyfile(t[0], t[1]); return True
        except Exception:
            return False
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        ok = list(tqdm(ex.map(_cp, tasks), total=len(tasks), desc="copy"))
    print(f" Copied {sum(ok)}/{len(tasks)} in {time.time()-t0:.0f}s")
else:
    print(f" Local cache complete ({n_dst} files)")

classes = sorted([d for d in os.listdir(LOCAL_DATA)
                  if os.path.isdir(os.path.join(LOCAL_DATA, d))])
NUM_CLASSES     = len(classes)
CLASS_TO_IDX    = {c: i for i, c in enumerate(classes)}
healthy_classes = [c for c in classes if 'healthy' in c.lower()]
print(f"Classes ({NUM_CLASSES}): {classes}")

dv1_path = os.path.join(OUT, 'checkpoints', 'dino_vits16.pth')
if not os.path.exists(dv1_path):
    os.makedirs(os.path.dirname(dv1_path), exist_ok=True)
    !wget -q -O {dv1_path} https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth

dino_v1 = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
ckpt = torch.load(dv1_path, map_location='cpu')
sd = {k.replace('module.','').replace('backbone.','').replace('encoder.',''): v
      for k, v in ckpt.items() if 'head' not in k}
dino_v1.load_state_dict(sd, strict=False)
dino_v1.eval().to(device)
for p in dino_v1.parameters():
    p.requires_grad = False

_dv1_store = {}
dv1_last_attn = dino_v1.blocks[-1].attn

def _dv1_patched_forward(x, **kwargs):
    B, N, C = x.shape
    qkv = dv1_last_attn.qkv(x).reshape(B, N, 3, dv1_last_attn.num_heads,
                                       C // dv1_last_attn.num_heads)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    scale = (C // dv1_last_attn.num_heads) ** -0.5
    cls_q = q[:, :, 0:1, :]
    cls_attn = ((cls_q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    _dv1_store['cls_attn'] = cls_attn[:, :, 0, 1:].detach().cpu()
    attn_full = ((q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    x = (attn_full @ v).transpose(1, 2).reshape(B, N, C)
    x = dv1_last_attn.proj(x)
    x = dv1_last_attn.proj_drop(x)
    return x

dv1_last_attn.forward = _dv1_patched_forward

tf_dv1 = T.Compose([T.Resize((224,224)), T.ToTensor(),
                    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_train = T.Compose([T.Resize((224,224)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                      T.RandomRotation(20), T.ColorJitter(0.2,0.2,0.2), T.ToTensor(),
                      T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_test = T.Compose([T.Resize((224,224)), T.ToTensor(),
                     T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

DV1_GRID = 14
N_HEADS  = 6
topk     = max(1, int(0.1 * DV1_GRID * DV1_GRID))

with torch.no_grad():
    _ = dino_v1(torch.randn(1,3,224,224).to(device))
print(f"Attention hook OK: {_dv1_store['cls_attn'].shape}")

all_images = []
for cls_name in classes:
    cls_dir = os.path.join(LOCAL_DATA, cls_name)
    for fname in sorted(os.listdir(cls_dir)):
        if fname.lower().endswith(('.jpg','.jpeg','.png')):
            all_images.append({
                'path': os.path.join(cls_dir, fname),
                'class': cls_name,
                'class_idx': CLASS_TO_IDX[cls_name],
                'is_diseased': 0 if cls_name in healthy_classes else 1,
            })

class SimpleDS(Dataset):
    def __init__(self, imgs, labels, transform):
        self.imgs = imgs; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        return self.transform(Image.open(self.imgs[idx]['path']).convert('RGB')), self.labels[idx]

class TestDS(Dataset):
    def __init__(self, imgs, transform):
        self.imgs = imgs; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        return (self.transform(Image.open(self.imgs[idx]['path']).convert('RGB')),
                self.imgs[idx]['is_diseased'])

n_dis = sum(x['is_diseased'] for x in all_images)
print(f"\n Ready: {len(all_images)} images | Diseased: {n_dis}, Healthy: {len(all_images)-n_dis}")

In [ ]:
import os, shutil, sys
from google.colab import files

NEEDED = ["leakage_audit.py", "shape_confound.py", "uahs_scale_invariant.py"]
missing = [f for f in NEEDED if not os.path.exists(f"/content/{f}")]

if missing:
    print("Select these (multi-select with ctrl/cmd):", missing)
    up = files.upload()
    for name in up:
        dest = f"/content/{os.path.basename(name)}"
        if os.path.abspath(name) != os.path.abspath(dest):
            shutil.move(name, dest)

for m in ["leakage_audit", "shape_confound", "uahs_scale_invariant"]:
    sys.modules.pop(m, None)

import leakage_audit as la
import shape_confound as sc
import uahs_scale_invariant as si

assert hasattr(la, "grouped_split"), "wrong leakage_audit.py"
assert hasattr(sc, "leaf_mask"),     "wrong shape_confound.py"
assert hasattr(si, "extract"),       "wrong uahs_scale_invariant.py"
print("All three modules OK")

In [ ]:
report, uf, imgs = la.run_audit(
    data_root      = LOCAL_DATA,
    seed           = 42, test_frac = 0.2,
    phash_thresh   = 0, dhash_thresh = 0,
    use_embeddings = False,          # faster; changes the split negligibly
    out_json       = os.path.join(RESULTS_DIR, "leakage_audit.json"),
)
print("\nuf + imgs ready")

In [ ]:
import matplotlib.pyplot as plt
import random as _r
from PIL import Image
import shape_confound as sc

_r.seed(0)
samp = _r.sample(all_images, 8)
fig, axes = plt.subplots(2, 8, figsize=(20, 5.5))
for i, rec in enumerate(samp):
    img = Image.open(rec['path']).convert('RGB')
    m = sc.leaf_mask(img)
    axes[0, i].imshow(img.resize((224, 224))); axes[0, i].axis('off')
    axes[0, i].set_title(rec['class'][:12], fontsize=8)
    if m is None:
        axes[1, i].text(.5, .5, 'FAILED', ha='center', color='red')
    else:
        axes[1, i].imshow(m, cmap='gray')
        axes[1, i].set_title(f"area={m.mean():.3f}", fontsize=8)
    axes[1, i].axis('off')
plt.suptitle("Leaf segmentation check — masks MUST cover the leaf, not the backdrop",
             fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# =====================================================================
# CORRECTED PIPELINE -- scale-invariant UAHS
# Run after Cell A + the audit (G1/G2, giving `uf` and `imgs`).
# Upload: shape_confound.py, uahs_scale_invariant.py, leakage_audit.py
# ~25 min on GPU.
# =====================================================================

import os, json, random
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from PIL import Image

import shape_confound as sc
import uahs_scale_invariant as si
import leakage_audit as la

assert 'uf' in dir() and 'imgs' in dir(), "run the audit cell (G2) first"

# ---------------------------------------------------------------------
# STEP 1 -- extract scale-invariant stats for every image (one pass, ~8 min)
# ---------------------------------------------------------------------
STATS, AREA, Y, CLS, KEEP, STAT_NAMES = si.extract(
    all_images, dino_v1, tf_dv1, _dv1_store, device,
    DV1_GRID, sc.leaf_mask, desc="scale-invariant stats")

print(f"\nusable images: {len(Y)} / {len(all_images)}")
print(f"stats per head: {STAT_NAMES}")

np.savez(os.path.join(RESULTS_DIR, 'scale_invariant_stats.npz'),
         STATS=STATS, AREA=AREA, Y=Y, CLS=CLS, names=np.array(STAT_NAMES))

# ---------------------------------------------------------------------
# STEP 2 -- which statistic actually carries pathology signal?
#           (labels used ONLY for this diagnostic, not for the pipeline)
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 2  Which scale-invariant statistic separates healthy/diseased?")
print("=" * 70)
print(f"  [control] leaf AREA alone         : AUC {roc_auc_score(Y, -AREA):.4f}"
      f"   <- the confound")
print(f"{'statistic':<16s} {'best head':>10s} {'AUC':>8s}")
print("-" * 40)
best = (None, None, 0.5)
for si_i, name in enumerate(STAT_NAMES):
    aucs = [roc_auc_score(Y, STATS[:, h, si_i]) for h in range(STATS.shape[1])]
    aucs = [max(a, 1 - a) for a in aucs]          # direction-agnostic
    h_best = int(np.argmax(aucs))
    print(f"{name:<16s} {h_best:>10d} {aucs[h_best]:>8.4f}")
    if aucs[h_best] > best[2]:
        best = (si_i, h_best, aucs[h_best])
STAT_IDX = best[0]
print(f"\n  -> using '{STAT_NAMES[STAT_IDX]}' (best single-head AUC {best[2]:.4f})")

# ---------------------------------------------------------------------
# STEP 3 -- UAHS on the scale-invariant statistic (NO labels)
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 3  UAHS head selection (scale-invariant, no labels)")
print("=" * 70)

results = []
N_SEEDS = 3
for seed in range(N_SEEDS):
    print(f"\n--- seed {seed+1}/{N_SEEDS} ---")
    random.seed(seed * 1000 + 42)

    train_s, test_s = la.grouped_split(uf, KEEP, seed=seed * 1000 + 42, test_frac=0.2)
    tr_ids = {id(r) for r in train_s}
    tr_mask = np.array([id(r) in tr_ids for r in KEEP])
    te_mask = ~tr_mask

    # --- UAHS: rank heads by variance of the statistic over an unlabelled pool
    pool = np.random.RandomState(seed).choice(
        np.where(tr_mask)[0], min(200, tr_mask.sum()), replace=False)
    sel_heads, head_scores = si.uahs_select(STATS, STAT_IDX, pool, k=2)
    print(f"  UAHS heads: {sel_heads}   (variances: "
          f"{np.round(head_scores, 4).tolist()})")

    # --- score = mean of the statistic over selected heads
    score = STATS[:, sel_heads, STAT_IDX].mean(axis=1)
    # orient so that HIGH = diseased, using only the unlabelled prior that
    # diseased leaves should be MORE concentrated
    if roc_auc_score(Y[tr_mask], score[tr_mask]) < 0.5:
        score = -score

    # --- pseudo-labels via the prevalence threshold (no labels consulted)
    healthy_frac = (Y[tr_mask] == 0).mean()          # assumed known (field walk)
    pl, thr = si.pseudo_labels(score[tr_mask], healthy_frac)
    p_acc = accuracy_score(Y[tr_mask], pl)
    p_auc = roc_auc_score(Y[tr_mask], score[tr_mask])
    print(f"  pseudo-label acc {p_acc:.4f} | teacher score AUC {p_auc:.4f}")

    # --- train the student on those pseudo-labels
    tr_recs = [KEEP[i] for i in np.where(tr_mask)[0]]
    te_recs = [KEEP[i] for i in np.where(te_mask)[0]]
    loader_t = DataLoader(SimpleDS(tr_recs, pl.tolist(), tf_train),
                          batch_size=64, shuffle=True, num_workers=2)
    loader_e = DataLoader(TestDS(te_recs, tf_test),
                          batch_size=64, shuffle=False, num_workers=2)

    model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
    model = model.to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)
    crit = nn.CrossEntropyLoss()

    model.train()
    for ep in range(30):
        for xb, yb in loader_t:
            xb, yb = xb.to(device), yb.to(device).long()
            loss = crit(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        if (ep + 1) % 10 == 0:
            print(f"    epoch {ep+1}/30")

    model.eval(); probs = []
    with torch.no_grad():
        for xb, _ in loader_e:
            probs.append(torch.softmax(model(xb.to(device)), 1)[:, 1].cpu().numpy())
    probs = np.concatenate(probs)
    y_te = Y[te_mask]
    auc = roc_auc_score(y_te, probs)
    acc = accuracy_score(y_te, (probs > 0.5).astype(int))
    f1 = f1_score(y_te, (probs > 0.5).astype(int))
    print(f"  * student AUC {auc:.4f}  acc {acc:.4f}  F1 {f1:.4f}")

    results.append({'seed': seed, 'heads': sel_heads,
                    'teacher_auc': float(p_auc), 'pseudo_acc': float(p_acc),
                    'student_auc': float(auc), 'student_acc': float(acc),
                    'student_f1': float(f1)})
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ---------------------------------------------------------------------
# STEP 4 -- the controls that make the result believable
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 4  Controls")
print("=" * 70)

# shape-only baseline on the same images
S_feat = []
for r in KEEP:
    m = sc.leaf_mask(Image.open(r['path']).convert('RGB'))
    S_feat.append(sc.shape_features(m) if m is not None else np.zeros(7))
S_feat = np.array(S_feat, dtype=np.float32)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
shape_auc = cross_val_score(
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    S_feat, Y, cv=cv, scoring='roc_auc').mean()

ta = [r['teacher_auc'] for r in results]
sa = [r['student_auc'] for r in results]
pa = [r['pseudo_acc'] for r in results]

print(f"  leaf AREA alone (confound)        : AUC {roc_auc_score(Y, -AREA):.4f}")
print(f"  shape-only baseline               : AUC {shape_auc:.4f}")
print(f"  --")
print(f"  teacher: scale-invariant attention: AUC {np.mean(ta):.4f} +/- {np.std(ta):.4f}")
print(f"  pseudo-label accuracy             :     {np.mean(pa):.4f} +/- {np.std(pa):.4f}")
print(f"  STUDENT (final)                   : AUC {np.mean(sa):.4f} +/- {np.std(sa):.4f}")
print(f"                                acc :     {np.mean([r['student_acc'] for r in results]):.4f}")
print(f"                                 F1 :     {np.mean([r['student_f1'] for r in results]):.4f}")
print("=" * 70)
if np.mean(sa) > shape_auc + 0.03:
    print("  VERDICT: the corrected pipeline beats the shape baseline. The signal")
    print("           is pathology, not leaf geometry.")
else:
    print("  VERDICT: the corrected pipeline does NOT clearly beat leaf geometry.")
    print("           Report this honestly -- it is the real finding.")
print("=" * 70)

with open(os.path.join(RESULTS_DIR, 'corrected_pipeline.json'), 'w') as f:
    json.dump({'runs': results,
               'statistic': STAT_NAMES[STAT_IDX],
               'area_only_auc': float(roc_auc_score(Y, -AREA)),
               'shape_only_auc': float(shape_auc),
               'teacher_auc_mean': float(np.mean(ta)), 'teacher_auc_std': float(np.std(ta)),
               'student_auc_mean': float(np.mean(sa)), 'student_auc_std': float(np.std(sa)),
               'pseudo_acc_mean': float(np.mean(pa)),
               'n_images': int(len(Y))}, f, indent=2)
print("saved corrected_pipeline.json")

In [ ]:
import os, shutil, sys
from google.colab import files

NEEDED = ["leakage_audit.py", "shape_confound.py",
          "uahs_scale_invariant.py", "feature_shape_control.py"]

missing = [f for f in NEEDED if not os.path.exists(f"/content/{f}")]

if missing:
    print("Select these files (ctrl/cmd-click to multi-select):")
    for f in missing:
        print("   -", f)
    up = files.upload()
    for name in up:
        dest = f"/content/{os.path.basename(name)}"
        if os.path.abspath(name) != os.path.abspath(dest):
            shutil.move(name, dest)

# clear any stale cached versions
for m in ["leakage_audit", "shape_confound",
          "uahs_scale_invariant", "feature_shape_control"]:
    sys.modules.pop(m, None)

import leakage_audit as la
import shape_confound as sc
import uahs_scale_invariant as si
import feature_shape_control as fsc

assert hasattr(la,  "grouped_split"),    "wrong/old leakage_audit.py"
assert hasattr(sc,  "leaf_mask"),        "wrong/old shape_confound.py"
assert hasattr(si,  "extract"),          "wrong/old uahs_scale_invariant.py"
assert hasattr(fsc, "extract_features"), "wrong/old feature_shape_control.py"

print("All four modules loaded OK")

In [ ]:
# =====================================================================
# CELL C-TEST — Do DINO FEATURES carry disease signal beyond leaf shape?
# This decides whether the paper has a working method or a negative result.
# =====================================================================
import sys, os, json
import numpy as np

sys.modules.pop("feature_shape_control", None)
import feature_shape_control as fsc
import shape_confound as sc

# shape-only baseline measured earlier on this dataset
SHAPE_BASELINE = 0.792

# ---------------------------------------------------------------------
# A. FULL IMAGE features  (what the original pipeline effectively saw)
# ---------------------------------------------------------------------
Xf, Yf, CLSf, SHf, AREAf, KEEPf = fsc.extract_features(
    all_images, dino_v1, tf_dv1, device, sc.leaf_mask, mode="full")
r_full = fsc.report("FULL IMAGE features", Xf, Yf, SHf, shape_auc=SHAPE_BASELINE)

# ---------------------------------------------------------------------
# B. LEAF-CROP NORMALISED features  (THE FIX: leaf scale removed by
#    construction — crop to the leaf bbox, resize to 224x224, so camera
#    distance and leaf size cannot drive anything)
# ---------------------------------------------------------------------
Xc, Yc, CLSc, SHc, AREAc, KEEPc = fsc.extract_features(
    all_images, dino_v1, tf_dv1, device, sc.leaf_mask, mode="crop")
r_crop = fsc.report("LEAF-CROP NORMALISED features", Xc, Yc, SHc,
                    shape_auc=SHAPE_BASELINE)

# ---------------------------------------------------------------------
# VERDICT
# ---------------------------------------------------------------------
print("\n" + "=" * 72)
print("VERDICT")
print("=" * 72)
print(f"  leaf shape alone (the confound)      : {SHAPE_BASELINE:.4f}")
print("  --")
print(f"  crop-normalised, UNSUPERVISED (2 lbl): {r_crop['unsup_auc']:.4f}")
print(f"  crop-normalised, shape-residualised  : {r_crop['probe_residualised']:.4f}")
print(f"  crop-normalised, shape-matched pairs : {r_crop['probe_matched']:.4f}")
print("-" * 72)

clears = (r_crop['probe_residualised'] > 0.75 and r_crop['probe_matched'] > 0.70)
if clears:
    print("  FEATURES CARRY REAL PATHOLOGY SIGNAL beyond leaf geometry.")
    print("  -> The pipeline is salvageable: cluster crop-normalised features.")
    print("     Camera distance can no longer drive the result.")
else:
    print("  Features do NOT clearly exceed leaf geometry.")
    print("  -> The honest finding is a negative result. Still publishable, and")
    print("     genuinely useful to everyone using background-controlled leaf data.")
print("=" * 72)

# save for the next stage
np.savez(os.path.join(RESULTS_DIR, 'crop_features.npz'),
         X=Xc, Y=Yc, CLS=CLSc, SHAPE=SHc)
with open(os.path.join(RESULTS_DIR, 'feature_shape_control.json'), 'w') as f:
    json.dump({'full': r_full, 'crop': r_crop,
               'shape_baseline': SHAPE_BASELINE}, f, indent=2, default=float)
print("\nsaved crop_features.npz + feature_shape_control.json")

In [ ]:
import os, shutil, sys
from google.colab import files

if not os.path.exists("/content/overcluster_pipeline.py"):
    print("Select overcluster_pipeline.py (download it from the chat first):")
    up = files.upload()
    for name in up:
        dest = f"/content/{os.path.basename(name)}"
        if os.path.abspath(name) != os.path.abspath(dest):
            shutil.move(name, dest)

sys.modules.pop("overcluster_pipeline", None)
import overcluster_pipeline as oc

assert hasattr(oc, "run") and hasattr(oc, "_margin_auc"), \
    "wrong/old file — run  !rm /content/overcluster_pipeline.py  and retry"
print("Module OK:", oc.__file__)

In [ ]:
import os, json
import numpy as np
import overcluster_pipeline as oc

d = np.load(os.path.join(RESULTS_DIR, 'crop_features.npz'), allow_pickle=True)
X, Y_bin, CLS, SHAPE = d['X'], d['Y'], d['CLS'], d['SHAPE']

cls_names   = sorted(set(CLS.tolist()))
name2idx    = {c: i for i, c in enumerate(cls_names)}
y_cls       = np.array([name2idx[c] for c in CLS])
healthy_idx = [i for i, c in enumerate(cls_names) if 'healthy' in c.lower()][0]
print("classes:", cls_names)
print("healthy index:", healthy_idx, "| n =", len(Y_bin))

res = oc.run(
    X             = X,
    y_cls         = y_cls,
    y_bin         = Y_bin,
    class_names   = cls_names,
    healthy_idx   = healthy_idx,
    k_list        = (7, 10, 14, 20, 30),   # over-clustering sweep
    n_per_cluster = 1,                     # ONE expert label per cluster
    seeds         = (0, 1, 2),
    shape_feats   = SHAPE,                 # control, run through the same pipeline
)

with open(os.path.join(RESULTS_DIR, 'overcluster_pipeline.json'), 'w') as f:
    json.dump(res, f, indent=2, default=float)
print("\nsaved overcluster_pipeline.json")

In [ ]:
import os, sys, json
import numpy as np
from google.colab import files
import shutil

if not os.path.exists("/content/select_k_and_stability.py"):
    print("Select select_k_and_stability.py:")
    up = files.upload()
    for name in up:
        dest = f"/content/{os.path.basename(name)}"
        if os.path.abspath(name) != os.path.abspath(dest):
            shutil.move(name, dest)
sys.modules.pop("select_k_and_stability", None)
import select_k_and_stability as sk

d = np.load(os.path.join(RESULTS_DIR, 'crop_features.npz'), allow_pickle=True)
X, Y_bin, CLS, SHAPE = d['X'], d['Y'], d['CLS'], d['SHAPE']
cls_names   = sorted(set(CLS.tolist()))
y_cls       = np.array([cls_names.index(c) for c in CLS])
healthy_idx = [i for i, c in enumerate(cls_names) if 'healthy' in c.lower()][0]

# ---- 1. choose k WITHOUT labels ----
selk = sk.select_k(X, k_list=(5, 7, 10, 12, 14, 16, 20, 25, 30), n_init=50)
K_AUTO = selk['consensus_k']

# ---- 2. stability at the LABEL-FREE k (the number we will report) ----
print("\n" + "="*74)
print(f"STABILITY at the label-free k = {K_AUTO}")
print("="*74)
stab_auto = sk.stability(X, y_cls, Y_bin, healthy_idx, K_AUTO,
                         seeds=range(10), n_init=500)

# ---- 3. for transparency, also at k=14 (the peeked-at best) ----
print("\n" + "="*74)
print("For comparison only — k=14 (chosen by looking at the answer; NOT reported")
print("as the headline, shown so the reader can see the difference)")
print("="*74)
stab_14 = sk.stability(X, y_cls, Y_bin, healthy_idx, 14,
                       seeds=range(10), n_init=500)

# ---- 4. shape control at the same label-free k ----
print("\n" + "="*74)
print(f"SHAPE CONTROL at k={K_AUTO} (identical pipeline, leaf geometry only)")
print("="*74)
stab_shape = sk.stability(SHAPE, y_cls, Y_bin, healthy_idx, K_AUTO,
                          seeds=range(10), n_init=500)

print("\n" + "="*74)
print("HEADLINE (label-free k, no cherry-picking)")
print("="*74)
print(f"  k = {K_AUTO}  ->  {K_AUTO} expert labels")
print(f"  binary AUC   : {stab_auto['auc_mean']:.4f} +/- {stab_auto['auc_std']:.4f}")
print(f"  binary acc   : {stab_auto['acc_mean']:.4f}")
print(f"  binary F1    : {stab_auto['f1_mean']:.4f}")
print(f"  7-class acc  : {stab_auto['cls_acc_mean']:.4f}")
print(f"  SHAPE control: {stab_shape['auc_mean']:.4f} +/- {stab_shape['auc_std']:.4f}")
print("="*74)

with open(os.path.join(RESULTS_DIR, 'select_k_stability.json'), 'w') as f:
    json.dump({'select_k': selk, 'k_auto': K_AUTO,
               'stability_auto': stab_auto, 'stability_k14': stab_14,
               'shape_control': stab_shape}, f, indent=2, default=float)
print("saved select_k_stability.json")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil, json
import numpy as np
from google.colab import files

OUT         = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
RESULTS_DIR = f"{OUT}/results"
FIGURES_DIR = f"{OUT}/figures"

if not os.path.exists("/content/few_shot_curve.py"):
    print("Select few_shot_curve.py:")
    up = files.upload()
    for name in up:
        dest = f"/content/{os.path.basename(name)}"
        if os.path.abspath(name) != os.path.abspath(dest):
            shutil.move(name, dest)

sys.modules.pop("few_shot_curve", None)
import few_shot_curve as fs

d = np.load(os.path.join(RESULTS_DIR, 'crop_features.npz'), allow_pickle=True)
X, Y_bin, CLS, SHAPE = d['X'], d['Y'], d['CLS'], d['SHAPE']
cls_names   = sorted(set(CLS.tolist()))
y_cls       = np.array([cls_names.index(c) for c in CLS])
healthy_idx = [i for i, c in enumerate(cls_names) if 'healthy' in c.lower()][0]
print(f"loaded {X.shape} features | classes: {cls_names}")

In [ ]:
res = fs.curve(
    X, y_cls, Y_bin, cls_names, healthy_idx,
    shots       = (1, 2, 5, 10, 20, 50, 100),
    n_repeats   = 5,        # was 10 — halves the time, still gives error bars
    test_frac   = 0.3,
    shape_feats = SHAPE,
)
with open(os.path.join(RESULTS_DIR, 'few_shot_curve.json'), 'w') as f:
    json.dump(res, f, indent=2, default=float)
print("saved")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm opencv-python-headless

import os, json, random, copy, time, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from concurrent.futures import ThreadPoolExecutor
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

TEA_LEAF_ROOT = "/content/drive/MyDrive/Tea Leaf Dataset"
OUT           = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
RESULTS_DIR   = f"{OUT}/results"
FIGURES_DIR   = f"{OUT}/figures"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# ---- dataset -> local disk (parallel, resumable) ----
LOCAL_DATA = "/tmp/tea_leaf_local"
N_WORKERS  = 32

def _count_files(p):
    return sum(len(f) for _, _, f in os.walk(p)) if os.path.isdir(p) else 0

n_src, n_dst = _count_files(TEA_LEAF_ROOT), _count_files(LOCAL_DATA)
if n_dst < n_src:
    t0 = time.time(); tasks = []
    for root, _, files_ in os.walk(TEA_LEAF_ROOT):
        rel = os.path.relpath(root, TEA_LEAF_ROOT)
        dst_dir = LOCAL_DATA if rel == '.' else os.path.join(LOCAL_DATA, rel)
        os.makedirs(dst_dir, exist_ok=True)
        for f in files_:
            src, dst = os.path.join(root, f), os.path.join(dst_dir, f)
            if not os.path.exists(dst):
                tasks.append((src, dst))
    print(f"Copying {len(tasks)} files with {N_WORKERS} workers...")
    def _cp(t):
        try: shutil.copyfile(t[0], t[1]); return True
        except Exception: return False
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        ok = list(tqdm(ex.map(_cp, tasks), total=len(tasks), desc="copy"))
    print(f" Copied {sum(ok)}/{len(tasks)} in {time.time()-t0:.0f}s")
else:
    print(f" Local cache complete ({n_dst} files)")

classes = sorted([d for d in os.listdir(LOCAL_DATA)
                  if os.path.isdir(os.path.join(LOCAL_DATA, d))])
NUM_CLASSES     = len(classes)
CLASS_TO_IDX    = {c: i for i, c in enumerate(classes)}
healthy_classes = [c for c in classes if 'healthy' in c.lower()]
print(f"Classes ({NUM_CLASSES}): {classes}")

dv1_path = os.path.join(OUT, 'checkpoints', 'dino_vits16.pth')
if not os.path.exists(dv1_path):
    os.makedirs(os.path.dirname(dv1_path), exist_ok=True)
    !wget -q -O {dv1_path} https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth

dino_v1 = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
ckpt = torch.load(dv1_path, map_location='cpu')
sd = {k.replace('module.','').replace('backbone.','').replace('encoder.',''): v
      for k, v in ckpt.items() if 'head' not in k}
dino_v1.load_state_dict(sd, strict=False)
dino_v1.eval().to(device)
for p in dino_v1.parameters():
    p.requires_grad = False

_dv1_store = {}
dv1_last_attn = dino_v1.blocks[-1].attn

def _dv1_patched_forward(x, **kwargs):
    B, N, C = x.shape
    qkv = dv1_last_attn.qkv(x).reshape(B, N, 3, dv1_last_attn.num_heads,
                                       C // dv1_last_attn.num_heads)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    scale = (C // dv1_last_attn.num_heads) ** -0.5
    cls_q = q[:, :, 0:1, :]
    cls_attn = ((cls_q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    _dv1_store['cls_attn'] = cls_attn[:, :, 0, 1:].detach().cpu()
    attn_full = ((q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    x = (attn_full @ v).transpose(1, 2).reshape(B, N, C)
    x = dv1_last_attn.proj(x)
    x = dv1_last_attn.proj_drop(x)
    return x

dv1_last_attn.forward = _dv1_patched_forward

tf_dv1 = T.Compose([T.Resize((224,224)), T.ToTensor(),
                    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_train = T.Compose([T.Resize((224,224)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                      T.RandomRotation(20), T.ColorJitter(0.2,0.2,0.2), T.ToTensor(),
                      T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_test = T.Compose([T.Resize((224,224)), T.ToTensor(),
                     T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

DV1_GRID, N_HEADS = 14, 6
topk = max(1, int(0.1 * DV1_GRID * DV1_GRID))

with torch.no_grad():
    _ = dino_v1(torch.randn(1,3,224,224).to(device))
print(f"Attention hook OK: {_dv1_store['cls_attn'].shape}")

all_images = []
for cls_name in classes:
    cls_dir = os.path.join(LOCAL_DATA, cls_name)
    for fname in sorted(os.listdir(cls_dir)):
        if fname.lower().endswith(('.jpg','.jpeg','.png')):
            all_images.append({
                'path': os.path.join(cls_dir, fname), 'class': cls_name,
                'class_idx': CLASS_TO_IDX[cls_name],
                'is_diseased': 0 if cls_name in healthy_classes else 1})

class SimpleDS(Dataset):
    def __init__(self, imgs, labels, transform):
        self.imgs, self.labels, self.transform = imgs, labels, transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        return self.transform(Image.open(self.imgs[i]['path']).convert('RGB')), self.labels[i]

class TestDS(Dataset):
    def __init__(self, imgs, transform):
        self.imgs, self.transform = imgs, transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        return (self.transform(Image.open(self.imgs[i]['path']).convert('RGB')),
                self.imgs[i]['is_diseased'])

n_dis = sum(x['is_diseased'] for x in all_images)
print(f"\n Ready: {len(all_images)} images | Diseased: {n_dis}, Healthy: {len(all_images)-n_dis}")

In [ ]:
import os, shutil, sys
from google.colab import files

NEEDED = ["shape_confound.py", "cross_dataset_confound.py"]
missing = [f for f in NEEDED if not os.path.exists(f"/content/{f}")]

if missing:
    print("Select (ctrl/cmd-click for multiple):", missing)
    up = files.upload()
    for name in up:
        dest = f"/content/{os.path.basename(name)}"
        if os.path.abspath(name) != os.path.abspath(dest):
            shutil.move(name, dest)

for m in ["shape_confound", "cross_dataset_confound"]:
    sys.modules.pop(m, None)

import shape_confound as sc
import cross_dataset_confound as cdc
assert hasattr(sc, "leaf_mask") and hasattr(sc, "attn_scores"), "wrong shape_confound.py"
assert hasattr(cdc, "scan_dataset"), "wrong cross_dataset_confound.py"
print("Modules OK")

In [ ]:
import os
CVD = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study/cross_validation_datasets"

DATASETS = {
    "TeaLeafDiseaseBD":  LOCAL_DATA,
    "PlantVillage":      f"{CVD}/PlantVillage",
    "PlantDoc":          f"{CVD}/PlantDoc",
    "RiceLeafDiseaseBD": f"{CVD}/RiceLeafDiseaseBD",
    "AppleLeafDisease":  f"{CVD}/AppleLeafDisease",
    "DiaMOS_Pear":       f"{CVD}/DiaMOS_Pear",
    "RoCoLe":            f"{CVD}/RoCoLe_binary",
}

for name, root in DATASETS.items():
    if not os.path.isdir(root):
        print(f"{name:<20s} NOT FOUND: {root}"); continue
    subs = sorted(d for d in os.listdir(root)
                  if os.path.isdir(os.path.join(root, d)))
    has_healthy = any('healthy' in s.lower() for s in subs)
    n_img = sum(1 for _,_,fs in os.walk(root) for f in fs
                if f.lower().endswith(('.jpg','.jpeg','.png')))
    print(f"{name:<20s} {len(subs):>3d} folders | {n_img:>6d} imgs | "
          f"healthy={'YES' if has_healthy else 'NO '} | e.g. {subs[:2]}")

In [ ]:
import os

CVD = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study/cross_validation_datasets"

# --- fix RiceLeafDiseaseBD: classes live one level deeper ---
rice_root = f"{CVD}/RiceLeafDiseaseBD/Original images"
print("Rice subfolders:", sorted(os.listdir(rice_root))[:8] if os.path.isdir(rice_root) else "CHECK PATH")

# --- inspect PlantDoc naming so we can define 'healthy' correctly ---
pd_root = f"{CVD}/PlantDoc"
print("\nPlantDoc folders:")
for s in sorted(os.listdir(pd_root)):
    print("   ", s)

In [ ]:
import json, os
CVD = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study/cross_validation_datasets"

# PlantDoc: healthy = the bare "<crop> leaf" folders, no disease term
PLANTDOC_HEALTHY = [
    "apple leaf", "bell_pepper leaf", "blueberry leaf", "cherry leaf",
    "raspberry leaf", "soyabean leaf", "tomato leaf", "grape leaf",
]

DATASETS = {
    "TeaLeafDiseaseBD":  (LOCAL_DATA,                              ("healthy",)),
    "PlantVillage":      (f"{CVD}/PlantVillage",                   ("healthy",)),
    "RiceLeafBD":        (f"{CVD}/RiceLeafDiseaseBD/Original images", ("healthy",)),
    "RoCoLe":            (f"{CVD}/RoCoLe_binary",                  ("healthy",)),
    "PlantDoc (field)":  (f"{CVD}/PlantDoc",                       tuple(PLANTDOC_HEALTHY)),
}

rows = []
for name, (root, hk) in DATASETS.items():
    r = cdc.scan_dataset(root, name, dino_v1, tf_dv1, _dv1_store, device,
                         DV1_GRID, topk,
                         healthy_keywords=hk, n_per_class=200)
    if r:
        rows.append(r)

cdc.summarise(rows)
with open(os.path.join(RESULTS_DIR, 'cross_dataset_confound.json'), 'w') as f:
    json.dump(rows, f, indent=2, default=float)
print("saved")